In [111]:
from alerce.core import Alerce
import pandas as pd

alerce = Alerce()

In [112]:
ztf_ids = [
    "ZTF22abhwlnm", "ZTF22aadesjc", "ZTF22abfdzrv", "ZTF21abgkfzh",
    "ZTF23aatcsou", "ZTF23aadjssg", "ZTF18aaiwzie", "ZTF23aaekebt",
    "ZTF23aadbtou", "ZTF22abghrui", "ZTF21abiggqx", "ZTF23aaefpfb",
    "ZTF22aatwxrl", "ZTF23aaahnss", "ZTF22absuavp", "ZTF23aafgmaz",
    "ZTF23aaflptz", "ZTF23aatcola", "ZTF23aajestr", "ZTF23aahjdxa",
    "ZTF22abrbohu", "ZTF23aarzzwu", "ZTF22abzajwl", "ZTF23aagxvad", "ZTF23aaempzk"
]

SUPERNOVA_CLASSES = ['sn', 'snia', 'snibc', 'snii', 'slsn']

In [113]:
def check_supernova(obj_id):
    df = alerce.query_probabilities(oid=obj_id, format='pandas')
    
    # Split LC and stamp classifiers
    lc = df[df['classifier_name'].str.contains('lc_classifier')]
    stamp = df[df['classifier_name'].str.contains('stamp_classifier')]
    
    # Top SN from each
    lc_sn = lc[lc['class_name'].str.lower().isin(SUPERNOVA_CLASSES)]
    stamp_sn = stamp[stamp['class_name'].str.lower().isin(SUPERNOVA_CLASSES)]
    
    result = {'object_id': obj_id}
    
    if not lc_sn.empty:
        lc_top = lc_sn.loc[lc_sn['probability'].idxmax()]
        result['lc_class'] = lc_top['class_name']
        result['lc_prob'] = lc_top['probability']
    else:
        result['lc_class'] = None
        result['lc_prob'] = None

    if not stamp_sn.empty:
        stamp_top = stamp_sn.loc[stamp_sn['probability'].idxmax()]
        result['stamp_class'] = stamp_top['class_name']
        result['stamp_prob'] = stamp_top['probability']
    else:
        result['stamp_class'] = None
        result['stamp_prob'] = None

    # Determine if object is a supernova
    if lc_sn.empty and stamp_sn.empty:
        # Not a supernova
        top = df.loc[df['probability'].idxmax()]
        result.update({
            'is_supernova': False,
            'agreement': None,
            'probability': top['probability'],
            'most_likely_class': top['class_name']
        })
    else:
        # Supernova
        result.update({
            'is_supernova': True,
            'agreement': (result['lc_class'] == result['stamp_class']) if result['lc_class'] and result['stamp_class'] else None,
            'probability': ((result['lc_prob'] or 0) + (result['stamp_prob'] or 0)) / (2 if result['lc_prob'] and result['stamp_prob'] else 1),
            'most_likely_class': None
        })
    
    return result

In [114]:

results = [check_supernova(oid) for oid in ztf_ids]
summary = pd.DataFrame(results)

In [115]:
avg_row = pd.DataFrame({
    'object_id': ['AVERAGE'],
    'is_supernova': [None],
    'lc_class': [None],
    'lc_prob': [summary['lc_prob'].mean()],
    'stamp_class': [None],
    'stamp_prob': [summary['stamp_prob'].mean()]
})
summary = pd.concat([summary, avg_row], ignore_index=True)

pd.set_option('display.max_columns', None)
summary

,object_id,lc_class,lc_prob,stamp_class,stamp_prob,is_supernova,agreement,probability,most_likely_class
0,ZTF22abhwlnm,SLSN,0.65000,SN,0.544367,True,False,0.597183,None
1,ZTF22aadesjc,SLSN,0.68000,SN,0.732096,True,False,0.706048,None
2,ZTF22abfdzrv,SLSN,0.72400,SN,0.313880,True,False,0.518940,None
3,ZTF21abgkfzh,SLSN,0.41800,SN,0.566296,True,False,0.492148,None
4,ZTF23aatcsou,SLSN,0.61400,SN,0.334540,True,False,0.474270,None
5,ZTF23aadjssg,SNIbc,0.33400,SN,0.716764,True,False,0.525382,None
6,ZTF18aaiwzie,SNIa,0.41600,SN,0.553450,True,False,0.484725,None
7,ZTF23aaekebt,SNII,0.51000,SN,0.391327,True,False,0.450663,None
8,ZTF23aadbtou,SNII,0.36600,SN,0.665639,True,False,0.515820,None
9,ZTF22abghrui,SLSN,0.64800,SN,0.252840,True,False,0.450420,None
